In this notebook, we showcase how to use the improve retrieval performance using chunking.

In [1]:
import numpy as np
import torch
from transformers import pipeline

from kvpress import (
    ExpectedAttentionPress,
    KnormPress,
    ObservedAttentionPress,
    RandomPress,
    SnapKVPress,
    StreamingLLMPress,
    apply_per_layer_compression,
)

/mount/home/setup/.cache/pypoetry/virtualenvs/kvpress-PV_RntMw-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load the pipeline and data

In [2]:
# Load pipeline

device = "cuda:0"
ckpt = "meta-llama/Meta-Llama-3.1-8B-Instruct"
attn_implementation = "flash_attention_2"
pipe = pipeline("kv-press-text-generation", model=ckpt, device=device, torch_dtype="auto", model_kwargs={"attn_implementation":attn_implementation})

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  9.53it/s]


In [3]:
import datasets 

df = datasets.load_dataset("simonjegou/ruler", "4096")["test"].to_pandas()
df = df.loc[df["task"] == "niah_single_3"].reset_index(drop=True)

# Use the pipeline with a press

In [4]:
import logging
logging.basicConfig()
logging.getLogger().setLevel(logging.DEBUG)

In [5]:
# Pick a press with a compression ratio, you can run the following cells with different presses
compression_ratio = 0.3
press = KnormPress(compression_ratio)

In [6]:
# Run the pipeline on a single question
idx = 0
context = df.iloc[idx]["context"] 
question = df.iloc[idx]["question"] 
true_answer = df.iloc[idx]["answer"][0]

pred_answer = pipe(context, question=question, press=press, 
                   max_new_token=1000)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")
print(f"Correctly predicted: {true_answer in pred_answer}")

DEBUG:kvpress.pipeline:Context Length: 3774
DEBUG:kvpress.pipeline:Compressed Context Length: 2641


Question:   What is the special magic uuid for amused-quart mentioned in the provided text? 
Answer:     1ff49b78-8946-4e85-b59c-de66bacfb3d0
Prediction: The special magic uuid for amused-quart is: 1ff49b78-8946-4e85-b59d-e6b9b3c3d0
Correctly predicted: False


In [7]:
# Run the pipeline on a single question
idx = 0
context = df.iloc[idx]["context"] 
question = df.iloc[idx]["question"] 
true_answer = df.iloc[idx]["answer"][0]

pred_answer = pipe(context, question=question, press=press, 
                   chunk_size=1000,
                   max_new_token=1000)["answer"]

print(f"Question:   {question}")
print(f"Answer:     {true_answer}")
print(f"Prediction: {pred_answer}")
print(f"Correctly predicted: {true_answer in pred_answer}")

DEBUG:kvpress.pipeline:Context Length: 3774
DEBUG:kvpress.pipeline:Compressed Context Length: 2641


Question:   What is the special magic uuid for amused-quart mentioned in the provided text? 
Answer:     1ff49b78-8946-4e85-b59c-de66bacfb3d0
Prediction: The special magic uuid for amused-quart is: 1ff49b78-8946-4e85-b59c-de66bacfb3d0
Correctly predicted: True
